# Phases 4-5 — smoke test + pilot (Colab)

Needs the Phase 3 embeddings already in Drive (`embed_colab.ipynb`).

1. **Smoke** — one cell (C1, NLI, linear, seed 0) end to end; test accuracy
   should land in the ~0.75-0.85 range for frozen mpnet + a linear head.
2. **Pilot** — 3 seeds x linear head x 10 conditions x 3 datasets = 90 cells.
   Sanity checks only, no statistics (PROTOCOL.md §17 phase 5).

Runs on CPU; a GPU runtime is fine too and a little faster. `run_grid.py` is
resumable, so a disconnect just means re-running the last cell.

Output `pilot_runs.parquet` is tiny — download it and commit to the repo; all
analysis (Phase 7) runs locally off that file.

In [ ]:
!git clone https://github.com/ryanteachman/sbert-head-ablation.git
%cd sbert-head-ablation
!pip install -q "pyarrow" "pyyaml" "scikit-learn"
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EMBED_DIR = '/content/drive/MyDrive/sbert-head-ablation/embeddings'
OUT_DIR   = '/content/drive/MyDrive/sbert-head-ablation/results'
import os, json; os.makedirs(OUT_DIR, exist_ok=True)
print(json.dumps(json.load(open(f'{EMBED_DIR}/meta.json'))['splits'], indent=2))

## 1. Smoke test — one real cell

In [ ]:
!python src/run_grid.py --pilot --embed-dir "{EMBED_DIR}" \
  --out "{OUT_DIR}/_smoke.parquet" --datasets nli --conditions C1 --seeds 0

In [ ]:
import pandas as pd
s = pd.read_parquet(f'{OUT_DIR}/_smoke.parquet').iloc[0]
print(f"NLI C1 linear seed0  ->  test_acc={s.test_acc:.4f}  macro_f1={s.test_macro_f1:.4f}")
assert 0.65 < s.test_acc < 0.90, 'accuracy outside the expected frozen-mpnet range - investigate before the pilot'
print('smoke OK')

## 2. Pilot — 90 cells
PROTOCOL.md §17 checks: absolute numbers believable; ordering plausible
(C1/C2/C3 ≳ C0; C6/C7 < their with-`u,v` counterparts; C4≈C1, C9≈C2);
per-seed spread small; early stopping fires before the epoch ceiling.

In [ ]:
!python src/run_grid.py --pilot --embed-dir "{EMBED_DIR}" --out "{OUT_DIR}/pilot_runs.parquet"

In [ ]:
import pandas as pd
df = pd.read_parquet(f'{OUT_DIR}/pilot_runs.parquet')
print(df.groupby(['dataset','condition'])['test_acc'].agg(['mean','std']).round(4).to_string())
print('\nepoch ceiling hits (should be ~0):',
      int((df.epochs_trained == df.epochs_trained.max()).sum() and (~df.early_stopped).sum()))
df.to_csv(f'{OUT_DIR}/pilot_runs.csv', index=False)
print('download:', f'{OUT_DIR}/pilot_runs.parquet')

## Done
Download `pilot_runs.parquet` from Drive, drop it in `results/`, commit. Review
locally, then run the full grid (`run_grid.py` with no `--pilot`, 600 cells).